# Architektura systemu Software 3.0

## Cel: Przekształcenie prototypu z lekcji 2.3 w solidną, modularną architekturę MVP, która przetrwa zderzenie z rzeczywistością.

### Czego się nauczysz:
1. Zrozumiesz, dlaczego trzymanie całego kodu w jednym pliku to dług techniczny i jak podzielić projekt na warstwy *Engine*, *Memory* i *Skills*, aby oddzielić logikę przetwarzania od stanu wiedzy.
2. Nauczysz się budować `Tech UI` w *Gradio*, które służy jako narzędzie diagnostyczne typu X-Ray (debug mode) do podglądu surowego stanu pamięci i procesu decyzyjnego agenta w czasie rzeczywistym.
3. Użyjesz *Structured Output* z wykorzystaniem Pydantic, wymuszając na modelu zwracanie precyzyjnego obiektu JSON z oddzielonym wewnętrznym monologiem od odpowiedzi dla użytkownika.
4. Dowiesz się, jak przenieść logikę sterowania z instrukcji warunkowych w Pythonie bezpośrednio do szablonów Jinja2, tworząc dynamiczne prompty reagujące na fazę rozmowy.
5. Poznasz koncepcję Human Context Window i zobaczysz, jak odpowiednia struktura katalogów chroni zasoby poznawcze programisty przed chaosem informacyjnym.
6. Przetestujesz integralność infrastruktury za pomocą trybu Echo, weryfikując poprawność przepływu danych między frontendem a backendem przed implementacją właściwej inteligencji.
7. Zobaczysz, jak zastosować podejście MVP poprzez świadome hardkodowanie profilu użytkownika, co pozwala na natychmiastowe testowanie logiki agenta bez narzutu implementacji bazy danych.


Zacznę od kolejnej brutalnej prawdy o współczesnym Software 3.0: kod sam w sobie przestał być wartością. Wiem, to może brzmieć przerażająco, ale spójrzmy faktom w oczy. Kawałki logiki, funkcje, klasy – to wszystko można dzisiaj wygenerować w sekundy przy pomocy LLM. Prawdziwa wartość inżyniera przesunęła się w inne miejsce. Gdzie? **W stronę konceptualnego projektu i architektury.**

Zróbmy prosty test: jeśli weźmiesz kartkę papieru i ołówek, a następnie precyzyjnie rozpiszesz przepływ danych, warstwy pamięci i momenty decyzyjne – kod "napisze się sam". Jeśli jednak Twoja koncepcja jest dziurawa, żaden model AI tego nie uratuje.

### Problem "Ludzkiego okna kontekstowego"

W poprzedniej lekcji (2.3) mieliśmy wszystko w jednym Notebooku. Było miło i szybko. Ale gdy system (prompt) rośnie, pojawia się problem. Nie chodzi o limit tokenów modelu. Chodzi o limity **Twojego mózgu**.

My też mamy swoje "okno kontekstowe". Kiedy wrzucasz wszystko do jednego worka, szybko tracisz zdolność rozumienia skutków ubocznych swoich zmian. Dlatego budujemy strukturę. Nie dla estetyki, ale po to, by zarządzać Twoją uwagą poznawczą.



Czas na układanie rzeczy? Zaczynamy.

## Architektura produkcyjna

Zanim przejdziemy do implementacji, chcę Ci pokazać szerszy kontekst. To, co zaraz zbudujemy, to nie jest tylko ćwiczenie akademickie. To uproszczona wersja architektury *battle-tested*, którą wprost w DataWorkshop wdrażamy na produkcję.

Gdy patrzymy na profesjonalne systemy agentowe, musimy wyraźnie rozdzielić dwa światy. To rozróżnienie uratuje Cię przed "spaghetti code" i chaosem w zespole.

### Dwa światy: Software 1.0 vs Software 3.0

Wyobraź sobie diagram podzielony na dwie strefy.

### 1. Strefa "niebieska" (Software 1.0 - Klasyczne IT)
To jest ten nudny, ale niezbędny świat. Tutaj mamy standardowy frontend (np. React) i klasyczny backend. Co one robią?
*   obsługują logowanie i reset haseł
*   zarządzają subskrypcjami i mailami
*   ładnie wyglądają na telefonie

Czy my się tym zajmujemy w tym kursie? **Nie.** To są problemy rozwiązane dekadę temu. Często zajmuje się tym zupełnie inny zespół deweloperski. Między "Nami" (AI) a "Nimi" (IT) jest po prostu kontrakt (API). Backend IT wysyła żądanie, my odsyłamy wynik. Koniec tematu.

### 2. Strefa "brązowa" (Software 3.0 - Agentic AI)
To jest nasze królestwo. Tu znajduje się "mózg". W tej strefie mamy:
*   **Agenta AI** – głównego decydenta
*   **Pamięć** – kontekst krótko- i długoterminowy
*   **Narzędzia** – integracje ze światem
*   **LLM Backend** – warstwę API, która wystawia agenta na świat (taki most dla Software 1.0)

W tej lekcji, dla uproszczenia (MVP), pomijamy osobny "LLM Backend API". Nasz interfejs połączy się bezpośrednio z klasą Agenta. Ale pamiętaj, że finalnie to byłaby osobna mikro-usługa.

![](../images/architecture_software3.0.jpg)

## Tech UI: Twoja tajna broń

I tu dochodzimy do elementu, który wielu początkujących pomija, a który jest **absolutnie krytyczny**. Na diagramie nazywamy po prostu **Tech UI**.

O co chodzi?
Użytkownik końcowy chce widzieć "ładny czat" i gładkie animacje. Ale Ty, jako architekt Software 3.0, musisz zajrzeć pod maskę, żeby dokładnie wiedzieć co tam naprawdę pulsuje, iskrzy i pracuje w środku.

Tech UI (które zbudujemy w Gradio) to panel sterowania, który raczej **nigdy** nie trafia do klienta. Służy do:
1.  **Debugowania** – gdy klient zgłasza, że agent "zwariował", nie próbujesz tego odtwarzać na produkcyjnym frontendzie. Wchodzisz do Tech UI.
2.  **X-Ray (rentgen)** – widzisz to, co ukryte: surowy stan pamięci, ukryte przemyślenia agenta (reasoning), zużycie tokenów, wybrane narzędzia.
3.  **Symulacji** – możesz ręcznie podmienić stan pamięci i sprawdzić, jak zachowa się agent w specyficznych warunkach.

Dlatego w tym kursie nie bawimy się w CSS-y i Reacta. Używamy **Gradio**. Dlaczego? Bo Gradio pozwala postawić Tech UI w 5 minut i skupić się na inżynierii agenta, a nie na centrowaniu div-a. To jest właśnie ta różnica w podejściu. Budujemy system tak, aby mieć nad nim pełną kontrolę i wgląd w jego "wnętrzości". 


To teraz najwyższy czas przenieść tę teorię do struktury plików.

## Anatomia projektu

W świecie agentowym panuje obecnie "Dziki Zachód". Nie ma jednego, świętego standardu, jak układać pliki w projekcie AI (w przeciwieństwie np. do Django czy Reacta). Dlatego w tym programie, tak jak zwykle, kierujemy się jedną zasadą: inżynierskim zdrowym rozsądkiem. Tyle i aż tyle, zwykle sprawdza się w praktyce.

Zbudujemy architekturę opartą na przepływie danych, który składa się z kilku bloków:
1. `Memory Layer` – tu agent trzyma stan (profile, fakty)
2. `Engine Layer` – tu dzieje się "mielenie" danych (logika, sklejanie promptów)
3. `Skills Layer` – i tu mały spoiler: w tej lekcji stworzymy folder `skills`. To jest zalążek **pamięci proceduralnej**, o której jeszcze nie mówiliśmy. Na razie potraktuj to jako miejsce na konkretne umiejętności agenta. W lekcji 2.5 rozwinie skrzydła

W tej lekcji zrobimy to wszystko "na piechotę". Przeniesiemy kod ręcznie, kawałek po kawałku. Chcę, żebyś poczuł(a) ciężar każdej decyzji architektonicznej, zanim zaczniesz to automatyzować. Na koniec odpalimy UI, w którym wreszcie porozmawiasz ze swoim Tutorem, widząc jednocześnie, co dzieje się pod maską.

Najpierw musimy zrozumieć mapę terenu. W Software 3.0 struktura katalogów to nie tylko estetyka. To sposób, w jaki zarządzamy **kontekstem** i **uwagą** LLM.

Schemat, który wdrożymy. Jest to architektura *battle-tested*, którą możesz powielać w kolejnych projektach.

```text
ai_tutor/
├── app.py                  # UI (Gradio)
├── config.py               # Konfiguracja (Model, Keys)
├── engine/                 # ⚙️ MECHANIKA (To, co "mieli" dane)
│   ├── base_agent.py       # Abstrakcja
│   ├── tutor_agent.py      # Konkretna implementacja
│   ├── prompter.py         # Sklejacz promptów (Jinja2 logic)
│   └── ...
├── memory/                 # 🧠 STAN (To, co agent "wie")
│   ├── schemas/            # Pydantic Models (Core, Semantic, Episodic)
│   ├── templates/          # Jinja Templates (core.j2, semantic.j2)
│   └── logic/              # Manager pamięci (CRUD)
└── skills/                 # 🛠️ UMIEJĘTNOŚCI (To, co agent "robi")
    ├── point_a.py          # Logika Skill'a
    ├── point_a.j2          # Prompt Skill'a
    └── ...
```

## Generowanie szkieletu (Skeleton)

Zamiast klikać prawym przyciskiem myszy "Nowy folder" przez 15 minut, zrobimy to po inżyniersku. Użyjemy skryptu `bash`, który w ułamku sekundy postawi całą strukturę.

Dla tych, którzy rzadziej zaglądają do terminala, szybkie wyjaśnienie magii, która zaraz się zadzieje:
- `mkdir` (make directory) – tworzy nowy folder. Parametr `-p` oznacza "stwórz też foldery nadrzędne, jeśli ich nie ma" (czyli buduje całą ścieżkę naraz).
- `touch` – tworzy pusty plik (lub aktualizuje jego czas modyfikacji, jeśli istnieje). My używamy tego, by stworzyć puste "wydmuszki", które zaraz wypełnimy kodem.

### ⚠️ Ważna uwaga o bezpieczeństwie
Zwróć uwagę na zakomentowaną linię:
```bash
# rm -rf $PROJECT_NAME
```
`rm` to **remove**, a `-rf` to **recursive force**. To jest atomowy przycisk. Jeśli odkomentujesz tę linię, skrypt bez pytania usunie cały folder projektu i postawi go na nowo. Zostawiłem to zakomentowane dla Twojego bezpieczeństwa. Używaj tylko wtedy, gdy chcesz zresetować projekt do zera (total wipe).

Uruchom poniższą komórkę:

In [3]:
%%bash

# ============================================
# Skrypt generujący strukturę projektu AI Tutor
# Wszystkie pliki są puste (lub minimalne __init__.py)
# ============================================

PROJECT_NAME="ai_tutor"

echo "Creating project structure: $PROJECT_NAME"


# ☠️☠️ Usuń jeśli istnieje (opcjonalnie - odkomentuj jeśli chcesz)
rm -rf $PROJECT_NAME
# ☠️☠️

# Utwórz główny folder
mkdir -p $PROJECT_NAME
cp .env $PROJECT_NAME/
cd $PROJECT_NAME

# ============================================
# FOLDERY
# ============================================

mkdir -p engine
mkdir -p memory/schemas
mkdir -p memory/templates
mkdir -p memory/logic
mkdir -p skills
mkdir -p utils
mkdir -p templates

# ============================================
# PLIKI GŁÓWNE (puste)
# ============================================

touch app.py
touch config.py

# ============================================
# ENGINE (puste)
# ============================================

touch engine/__init__.py
touch engine/client.py
touch engine/base_agent.py
touch engine/tutor_agent.py
touch engine/prompter.py
touch engine/handlers.py
touch engine/jobs.py

# ============================================
# MEMORY - SCHEMAS (puste)
# ============================================

touch memory/__init__.py
touch memory/schemas/__init__.py
touch memory/schemas/core.py
touch memory/schemas/working.py
touch memory/schemas/workflow.py
touch memory/schemas/episodic.py
touch memory/schemas/semantic.py
touch memory/schemas/resources.py

# ============================================
# MEMORY - TEMPLATES (puste .j2)
# ============================================

touch memory/templates/core.j2
touch memory/templates/episodic.j2
touch memory/templates/semantic.j2

# ============================================
# MEMORY - LOGIC (puste)
# ============================================

touch memory/logic/__init__.py
touch memory/logic/manager.py
touch memory/logic/compressor.py

# ============================================
# SKILLS (puste)
# ============================================

touch skills/__init__.py
touch skills/_base.py
touch skills/point_a.py
touch skills/point_a.j2
touch skills/point_b.py
touch skills/point_b.j2
touch skills/plan.py
touch skills/plan.j2

# ============================================
# UTILS (puste)
# ============================================

touch utils/__init__.py
touch utils/helpers.py

# ============================================
# TEMPLATES (puste)
# ============================================

touch templates/main.j2


# ============================================
# PODSUMOWANIE
# ============================================

echo ""
echo "✅ Structure created!"
echo ""
echo "📁 Project tree:"
find . -type f | head -40
echo ""
echo "📊 Statistics:"
echo "   Folders: $(find . -type d | wc -l)"
echo "   Files:   $(find . -type f | wc -l)"


# ============================================
# GIT
# ============================================
git init
git add .
git config --global init.defaultBranch master
git commit -m "init project"


Creating project structure: ai_tutor

✅ Structure created!

📁 Project tree:
./templates/main.j2
./utils/helpers.py
./utils/__init__.py
./skills/point_a.j2
./skills/plan.j2
./skills/point_a.py
./skills/plan.py
./skills/__init__.py
./skills/point_b.py
./skills/_base.py
./skills/point_b.j2
./app.py
./config.py
./engine/tutor_agent.py
./engine/handlers.py
./engine/jobs.py
./engine/base_agent.py
./engine/prompter.py
./engine/__init__.py
./engine/client.py
./memory/templates/semantic.j2
./memory/templates/episodic.j2
./memory/templates/core.j2
./memory/schemas/workflow.py
./memory/schemas/core.py
./memory/schemas/working.py
./memory/schemas/resources.py
./memory/schemas/__init__.py
./memory/schemas/episodic.py
./memory/schemas/semantic.py
./memory/logic/manager.py
./memory/logic/compressor.py
./memory/logic/__init__.py
./memory/__init__.py
./.env



/opt/conda/envs/python310/lib/python3.10/json/encoder.py:257: UserWarning: date_default is deprecated since jupyter_client 7.0.0. Use jupyter_client.jsonutil.json_default.
  return _iterencode(o, 0)


📊 Statistics:
   Folders: 9
   Files:   35
Initialized empty Git repository in /home/jovyan/agentic_ai/module2/notebooks/ai_tutor/.git/



*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'jovyan@jupyter-wlazlo-2epiotr1-5f8639.(none)')


CalledProcessError: Command 'b'\n# ============================================\n# Skrypt generuj\xc4\x85cy struktur\xc4\x99 projektu AI Tutor\n# Wszystkie pliki s\xc4\x85 puste (lub minimalne __init__.py)\n# ============================================\n\nPROJECT_NAME="ai_tutor"\n\necho "Creating project structure: $PROJECT_NAME"\n\n\n# \xe2\x98\xa0\xef\xb8\x8f\xe2\x98\xa0\xef\xb8\x8f Usu\xc5\x84 je\xc5\x9bli istnieje (opcjonalnie - odkomentuj je\xc5\x9bli chcesz)\nrm -rf $PROJECT_NAME\n# \xe2\x98\xa0\xef\xb8\x8f\xe2\x98\xa0\xef\xb8\x8f\n\n# Utw\xc3\xb3rz g\xc5\x82\xc3\xb3wny folder\nmkdir -p $PROJECT_NAME\ncp .env $PROJECT_NAME/\ncd $PROJECT_NAME\n\n# ============================================\n# FOLDERY\n# ============================================\n\nmkdir -p engine\nmkdir -p memory/schemas\nmkdir -p memory/templates\nmkdir -p memory/logic\nmkdir -p skills\nmkdir -p utils\nmkdir -p templates\n\n# ============================================\n# PLIKI G\xc5\x81\xc3\x93WNE (puste)

### Co tu się właściwie wydarzyło? 

Jeśli patrzysz na tę listę plików i czujesz lekki niepokój ("Po co tego aż tyle?"), to spokojnie. To normalne. Wyjaśnijmy to sobie:

1.  **Puste pliki to tylko deklaracja intencji.** W tym momencie `app.py` czy `tutor_agent.py` mają rozmiar 0 bajtów. Nic w nich nie ma. To tak, jak postawisz puste regały w magazynie i przykleisz do nich etykiety "śruby", "nakrętki", "młotki". Jeszcze nic tam nie leży, ale już wiesz, gdzie szukać.
2.  **Magia `__init__.py`.** Zauważysz sporo plików o tej nazwie. Dla Pythona to sygnał: *"Hej, ten folder to nie jest zwykły katalog na zdjęcia z wakacji. Traktuj go jako pakiet (package), z którego można importować kod"*. Dzięki temu będziemy mogli pisać `from engine.tutor_agent import ...`.
3.  **Fizyczna mapa logicznej architektury.** To drzewo plików to dosłowne odzwierciedlenie diagramu, o którym mówiliśmy wcześniej. Folder `memory` to pamięć, `engine` to silnik, `skills` to umiejętności.

Teraz, gdy mamy już regały, czas zacząć układać na nich towar. Zaczynamy od najważniejszego elementu: konfiguracji.



## Struktura projektu

```
ai_tutor/
│
│   # ══════════════════════════════════════════════════════════
│   # WARSTWA WEJŚCIA
│   # ══════════════════════════════════════════════════════════
│
├── app.py                          # Gradio UI — punkt wejścia
├── config.py                       # API keys, model, stałe, feature flags
│
│   # ══════════════════════════════════════════════════════════
│   # SILNIK (engine) — mechanika działania agenta
│   # ══════════════════════════════════════════════════════════
│
├── engine/
│   ├── __init__.py                 # Re-eksport: from .tutor_agent import TutorAgent
│   │
│   ├── client.py                   # LLM Client wrapper
│   │                               #    - create_client() → OpenAI/Anthropic
│   │                               #    - call_llm(messages, output_model, stream) → response
│   │
│   ├── base_agent.py               # BaseAgent (abstrakcja)
│   │                               #    - _make_api_call(output_model, stream)
│   │                               #    - _handle_response(response, handler)
│   │
│   ├── tutor_agent.py              # TutorAgent (konkretna implementacja)
│   │                               #    - process_message(user_input) → response
│   │                               #    - run_skill(skill_name) → output
│   │                               #    - decide_next_action() → skill | response
│   │
│   ├── prompter.py                 # SystemPrompter
│   │                               #    - __init__(state, conversation_history)
│   │                               #    - prepare_system_prompt() → str
│   │                               #    - (używa memory/templates/ + templates/main.j2)
│   │
│   ├── handlers.py                 # Response Handlers
│   │                               #    - ResponseStreamHandler.run(response) → async generator
│   │                               #    - ResponseBlockHandler.run(response) → dict
│   │                               #    - (parsowanie chunków, delta, usage)
│   │
│   └── jobs.py                     # Background Jobs
│                                   #    - PostSessionJob.run(session_data)
│                                   #    - (kompresja epizodów, cleanup, analytics)
│
│   # ══════════════════════════════════════════════════════════
│   # PAMIĘĆ (memory) — wszystkie 7 warstw
│   # ══════════════════════════════════════════════════════════
│
├── memory/
│   ├── __init__.py                 # Re-eksport schematów
│   │
│   │   # ─────────────────────────────────────────────────────
│   │   # SCHEMATY (Pydantic) — CO przechowujemy
│   │   # ─────────────────────────────────────────────────────
│   ├── schemas/
│   │   ├── __init__.py
│   │   ├── core.py                 # 🏛️ CoreMemory
│   │   ├── working.py              # ⚡ WorkingMemory (conversation_buffer, temp_vars)
│   │   ├── workflow.py             # 🎯 WorkflowMemory (phase, milestone, homework)
│   │   ├── episodic.py             # 📚 EpisodeMemory, MistakeLog
│   │   ├── semantic.py             # 🧠 StudentProfile, PointA, PointB, Plan
│   │   └── resources.py            # 🔧 ResourceMemory (api_status, tokens)
│   │
│   │   # ─────────────────────────────────────────────────────
│   │   # SZABLONY (Jinja2) — JAK pokazujemy w prompcie
│   │   # ─────────────────────────────────────────────────────
│   ├── templates/
│   │   ├── core.j2                 # 🏛️ tożsamość, zasady
│   │   ├── episodic.j2             # 📚 historia, błędy
│   │   └── semantic.j2             # 🧠 profil, plan
│   │
│   │   # ─────────────────────────────────────────────────────
│   │   # LOGIKA — operacje na pamięci
│   │   # ─────────────────────────────────────────────────────
│   └── logic/
│       ├── __init__.py
│       ├── manager.py              # MemoryManager — LIFECYCLE
│       │                           #    - update_from_output(skill_output) → state
│       │                           #    - add_episode(episode) → state
│       │                           #    - validate(state) → bool
│       │                           #    - merge(old, new) → state
│       │                           #    - cleanup(state, max_episodes) → state
│       │
│       └── compressor.py           # EpisodeCompressor
│                                   #    - compress(conversation_log) → EpisodeMemory
│                                   #    - extract_mistakes(text) → List[MistakeLog]
│                                   #    - extract_facts(text) → List[str]
│
│   # ══════════════════════════════════════════════════════════
│   # SKILLS (🛠️ PROCEDURAL MEMORY)
│   # ══════════════════════════════════════════════════════════
│
├── skills/
│   ├── __init__.py                 # Re-eksport skills
│   ├── _base.py                    # Skill base class
│   │                               #    - name, description, output_model
│   │                               #    - get_prompt_template() → str
│   │                               #    - execute(context) → output_model
│   │
│   ├── point_a.py                  # PointASkill + PointAOutput
│   ├── point_a.j2
│   │
│   ├── point_b.py                  # PointBSkill + PointBOutput
│   ├── point_b.j2
│   │
│   ├── plan.py                     # PlanSkill + PlanOutput
│   └── plan.j2
│
│   # ══════════════════════════════════════════════════════════
│   # UTILITIES — helpery
│   # ══════════════════════════════════════════════════════════
│
├── utils/
│   ├── __init__.py
│   └── helpers.py                  # Funkcje pomocnicze
│                                   #    - format_sse_event(event, data) → str
│                                   #    - get_timestamp() → int
│                                   #    - escape_newlines(text) → str
│
│   # ══════════════════════════════════════════════════════════
│   # SZABLONY SYSTEMOWE
│   # ══════════════════════════════════════════════════════════
│
└── templates/
    └── main.j2                     # Główny orchestrator
                                    # {% include 'memory/templates/core.j2' %}
                                    # {% include 'memory/templates/semantic.j2' %}
                                    # etc.
```

## Przepływ danych (sekwencja)

```
┌─────────────────────────────────────────────────────────────────────┐
│  1. USER INPUT                                                      │
│     app.py (Gradio) → TutorAgent.process_message()                  │
└─────────────────────────────────────────────────────────────────────┘
                                    │
                                    ▼
┌─────────────────────────────────────────────────────────────────────┐
│  2. DECIDE ACTION                                                   │
│     TutorAgent.decide_next_action()                                 │
│        │                                                            │
│        ├── workflow.phase == ONBOARDING? → run_skill("point_a")     │
│        ├── needs_goal_clarification? → run_skill("point_b")         │
│        └── else → direct_response                                   │
└─────────────────────────────────────────────────────────────────────┘
                                    │
                                    ▼
┌─────────────────────────────────────────────────────────────────────┐
│  3. PREPARE PROMPT                                                  │
│     SystemPrompter.prepare_system_prompt()                          │
│        │                                                            │
│        ├── Load templates/main.j2                                   │
│        ├── Include memory/templates/*.j2                            │
│        ├── Inject current state (schemas)                           │
│        └── Return: system_prompt (str)                              │
└─────────────────────────────────────────────────────────────────────┘
                                    │
                                    ▼
┌─────────────────────────────────────────────────────────────────────┐
│  4. CALL LLM                                                        │
│     BaseAgent._make_api_call(output_model, stream)       │
│        │                                                            │
│        ├── client.call_llm(messages, output_model, stream)          │
│        └── Return: raw response                                     │
└─────────────────────────────────────────────────────────────────────┘
                                    │
                                    ▼
┌─────────────────────────────────────────────────────────────────────┐
│  5. HANDLE RESPONSE                                                 │
│     ResponseStreamHandler.run(response)  ─OR─  ResponseBlockHandler │
│        │                                                            │
│        ├── Parse chunks / full response                             │
│        ├── Yield events: stream_start, delta, content_finish, usage │
│        └── Return: parsed output (Pydantic model)                   │
└─────────────────────────────────────────────────────────────────────┘
                                    │
                                    ▼
┌─────────────────────────────────────────────────────────────────────┐
│  6. UPDATE MEMORY                                                   │
│     MemoryManager.update_from_output(skill_output)                  │
│        │                                                            │
│        ├── Validate output                                          │
│        ├── Update semantic (profile, point_a, point_b, plan)        │
│        ├── Update workflow (phase, milestone)                       │
│        ├── Update working (conversation_buffer)                     │
│        └── Return: updated state                                    │
└─────────────────────────────────────────────────────────────────────┘
                                    │
                                    ▼
┌─────────────────────────────────────────────────────────────────────┐
│  7. BACKGROUND (async, po sesji)                                    │
│     PostSessionJob.run(session_data)                                │
│        │                                                            │
│        ├── EpisodeCompressor.compress(conversation_log)             │
│        ├── MemoryManager.add_episode(episode)                       │
│        ├── MemoryManager.cleanup(max_episodes=10)                   │
│        └── Save state to persistence                                │
└─────────────────────────────────────────────────────────────────────┘
```

## Szybka wygrana: Odpalmy "Tech UI" (Echo Mode)

Mamy strukturę, ale na razie to tylko "cmentarzysko pustych plików". Ożywmy to.

Zależy mi na tym, żebyś jak najszybciej zobaczył(a) efekt. Dlatego teraz, zamiast mozolnie wypełniać wszystkie warstwy (pamięć, silnik, skille), zrobimy skrót (bypass). Podłączymy **Tech UI** (Gradio) bezpośrednio do prostej funkcji "Echo". Ty piszesz "Cześć", system odpisuje "Cześć".

To pozwoli nam zweryfikować, czy środowisko działa, i – co ważniejsze – zobaczysz szkielet naszego **Panelu Sterowania**.

### Jak będziemy edytować pliki?

Jesteśmy w Notebooku, więc masz dwie opcje:

1.  **Klasyczna (GUI):** Możesz klikać w panelu plików po lewej stronie, otwierać plik `ai_tutor/app.py` i wklejać kod. Działa, ale wymaga klikania.

![](../images/jupyter_lab_structure.jpg)

2.  **Prostsza (Notebook Magic):** Użyjemy komendy `%%writefile path/to/file.py`. To pozwoli nam nadpisać zawartość pliku bezpośrednio z poziomu komórki notebooka.

W tym kursie będę używał metody nr 2. Jest szybsza, powtarzalna i od razu widać w lekcji, co wylądowało w pliku.

### Implementacja interfejsu (app.py)

Zdefiniujmy nasz `app.py`. Zwróć uwagę na strukturę `gr.Blocks`. To nie jest zwykły czat. Dzielimy ekran na dwie kolumny:
*   **Lewa:** Interfejs konwersacyjny.
*   **Prawa:** "Rentgen" (X-Ray) stanu. Na razie będzie pokazywał statyczny JSON, ale to tu wkrótce będziesz podglądać myśli swojego agenta.

Uruchom poniższą komórkę:

In [ ]:
%%writefile ai_tutor/app.py
import gradio as gr

# ==========================================
# 1. DUMMY ENGINE (Na razie tylko echo)
# ==========================================
def echo_logic(message, history):
    return f"ECHO: {message}"

# ==========================================
# 2. TECH UI LAYOUT
# ==========================================
with gr.Blocks(title="AI Tutor - Tech UI", theme=gr.themes.Soft()) as demo:
    
    gr.Markdown("# AI Tutor - Tech UI (v0.1)")
    
    with gr.Row():
        # --- KOLUMNA LEWA: INTERFEJS ROZMOWY ---
        with gr.Column(scale=6):
            gr.Markdown("### 💬 Chat")
            # ChatInterface to gotowy komponent do obsługi historii rozmowy
            chat = gr.ChatInterface(
                fn=echo_logic, 
                type="messages" # Nowy format Gradio (list of dicts)
            )
            
        # --- KOLUMNA PRAWA: DEBUG & STATE (X-RAY) ---
        with gr.Column(scale=4, variant="panel"):
            gr.Markdown("### Internal State")
            
            # To tutaj będziemy wrzucać "wnętrzości" agenta
            debug_view = gr.JSON(
                value={
                    "status": "SYSTEM_BOOT",
                    "engine": "EchoMode",
                    "memory_usage": "0%"
                }, 
                label="Live State Dump"
            )
            
            gr.Markdown("> **Info:** This is an engineering view. The client does not see this")


# ==========================================
# 3. URUCHOMIENIE
# ==========================================
if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=8080)

### Uruchomienie systemu

Teraz moment prawdy. Ponieważ pracujemy na zdalnym serwerze, nie możesz po prostu wejść na `localhost`. Musimy użyć specjalnego przekierowania (proxy).

1.  Uruchom poniższą komórkę (zobaczysz `[*]` oznaczające, że serwer działa).
2.  **Poczekaj chwilę**, aż zobaczysz komunikat w logach `Running on local URL...`.
3.  Kliknij w duży link poniżej kodu.

**Uwaga:** Gradio zablokuje wykonanie komórki (zobaczysz `[*]` po lewej stronie). To normalne – serwer nasłuchuje. Aby przejść dalej w lekcji, będziesz musiał(a) zatrzymać komórkę (przycisk Stop ⏹️), ale na razie pobawmy się chwilę.

In [ ]:
!python ai_tutor/app.py

Gdy skrypt powyżej będzie już działał ("mielił"), kliknij tutaj, aby otworzyć UI w nowej karcie:

## 👉👉 [Klik =>  TECH UI (Port 8080)](/hub/user-redirect/proxy/8080/) 👈👈

### Co zobaczysz?
1.  Tytuł **"AI Tutor - Tech UI"**
2.  Po lewej stronie czat – wpisz "Test", powinien odpisać "ECHO: Test"
3.  Po prawej stronie panel JSON ze statusem systemu (na razie statyczny).


### Co powinieneś zobaczyć?
1.  Tytuł: "AI Tutor - Tech UI".
2.  Czat (Lewa strona): Napisz cokolwiek, np. "Cześć". System odpisze dokładnie to samo: "ECHO: Cześć".
    *   **Co to jest Echo?** To najprostszy możliwy test w IT. Działa jak odbicie w lustrze – system nie "myśli", tylko odsyła Twój tekst z powrotem. Jeśli to widzisz, to znaczy, że **rury są szczelne**: Twoja wiadomość dotarła do serwera i wróciła, a interfejs działa poprawnie.
3.  Panel (prawa strona): Widzisz tam JSON ze statusem systemu. To Twój przyszły debug-mode (rentgen systemu).


![](../images/gradio_v0.1.jpg)

Działa? Świetnie. Masz już postawiony serwer, który "gada" i panel inżynierski, który "widzi".

### 🛑 WAŻNE: Zatrzymaj komórkę
Gradio blokuje działanie notebooka (zobaczysz `[*]` przy komórce z kodem). Aby przejść do dalszej części lekcji i zacząć pisać prawdziwy kod:
1.  Wróć do komórki powyżej.
2.  Naciśnij przycisk **Stop ⏹️** w menu Jupytera.
3.  Dopiero gdy `[*]` zniknie, przejdź dalej.

Teraz "wystarczy" tylko zamienić prostą funkcję `echo_logic` na prawdziwy mózg agenta. Bułka z masłem. No to zróbmy to krok po kroku.

## Fundamenty systemu (*boilerplate*)

Zanim przejdziemy do projektowania wyrafinowanej pamięci, musimy wylać beton pod fundamenty.

W Software 3.0, podobnie jak w klasycznym, mamy trochę kodu "technicznego" – konfiguracja, klient API, obsługa ścieżek. Nie ma sensu, aby tracić czas na ręczne wpisywanie `import os` czy ustawianie zmiennych środowiskowych. To są rzeczy powtarzalne.

Dlatego tę część zrobię za Ciebie. Twoim zadaniem będzie jedynie przejrzeć te pliki i je uruchomić. Nie musisz wszystkiego rozumieć w 100% — to na tym etapie naprawdę nie jest potrzebne. Wystarczy, że zaczniesz kojarzyć podstawowe rzeczy i powoli oswoisz się z tym, co tam się dzieje. Każdy zaczyna od małych kroków, a to już jest bardzo dobry start.

Ważniejsze od składni jest zrozumienie, **gdzie** co leży. 

Zobacz, tu jest mapa plików, które za moment wypełnimy treścią (reszta struktury jest ukryta dla czytelności):

```text
ai_tutor/
├── app.py                  <-- 🟢 UI (Gradio): Punkt wejścia do aplikacji
├── config.py               <-- 🟢 KONFIGURACJA: Klucze, ścieżki, model
├── engine/
│   ├── client.py           <-- 🟢 KLIENT do API (OpenAI + Instructor)
│   ├── prompter.py         <-- 🟢 PROMPTER: System sklejania szablonów Jinja2
│   ├── tutor_agent.py      <-- 🟢 AGENT: (Główny) Agent AI
│   └── ...
└── ...
```

## Konfiguracja (`config.py`)
To proste: wskazujemy, z jakiego modelu korzystamy i gdzie leżą nasze szablony.


```text
ai_tutor/
├── app.py                  <-- ⚪ UI (Gradio): Punkt wejścia do aplikacji
├── config.py               <-- 🟡 KONFIGURACJA: Klucze, ścieżki, model
├── engine/
│   ├── client.py           <-- ⚪ KLIENT do API (OpenAI + Instructor)
│   ├── prompter.py         <-- ⚪ PROMPTER: System sklejania szablonów Jinja2
│   ├── tutor_agent.py      <-- ⚪ AGENT: (Główny) Agent AI
│   └── ...
└── ...
```

In [ ]:
%%writefile ai_tutor/config.py
import os
from dotenv import load_dotenv

load_dotenv()

class Config:
    MODEL_NAME = "gpt-4o-mini" #model as config

    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    TEMPLATES_DIR = os.path.join(BASE_DIR, "templates")
    MEMORY_DIR = os.path.join(BASE_DIR, "memory")

### Klient API (`engine/client.py`)
Tutaj tworzymy połączenie ze światem zewnętrznym. Używamy `base_url` (bo działamy w specyficznym środowisku warsztatowym) oraz biblioteki `instructor`, która wymusi na modelu zwracanie JSON-a.


```text
ai_tutor/
├── app.py                  <-- ⚪ UI (Gradio): Punkt wejścia do aplikacji
├── config.py               <-- 🟢 KONFIGURACJA: Klucze, ścieżki, model
├── engine/
│   ├── client.py           <-- 🟡 KLIENT do API (OpenAI + Instructor)
│   ├── prompter.py         <-- ⚪ PROMPTER: System sklejania szablonów Jinja2
│   ├── tutor_agent.py      <-- ⚪ AGENT: (Główny) Agent AI
│   └── ...
└── ...
```

In [ ]:
%%writefile ai_tutor/engine/client.py
from functools import lru_cache
import instructor
from openai import OpenAI
from config import Config

@lru_cache()
def get_client():
    """
    Returns a configured OpenAI client with the Instructor patch.
    Thanks to @lru_cache, the client is created only once (singleton).
    """
    if not Config.OPENAI_API_KEY:
        raise ValueError("❌ Brak klucza API w konfiguracji!")

    client = OpenAI(
        base_url = "https://llm-api.dataworkshop.eu",
        api_key=Config.OPENAI_API_KEY
    )
    

    return instructor.patch(client, mode=instructor.Mode.MD_JSON)

## Prompter (`engine/prompter.py`)

To jest nasz "sklejacz". Jego jedynym zadaniem jest wziąć dane (profil, historia), wziąć szablon `.j2` i wygenerować gotowy tekst promptu.

Spójrz na poniższy schemat. Prompter to element wykonawczy, który realizuje logikę Bloku 4 (*Context Engineering*). Zbiera on wszystkie fragmenty "System Role" (profile, pamięć, instrukcje), aby dostarczyć do API finalną, wyrenderowaną wersję System Promptu (Blok 5).


![](../../module1/images/context_engineering.jpg)


```text
ai_tutor/
├── app.py                  <-- ⚪ UI (Gradio): Punkt wejścia do aplikacji
├── config.py               <-- 🟢 KONFIGURACJA: Klucze, ścieżki, model
├── engine/
│   ├── client.py           <-- 🟢 KLIENT do API (OpenAI + Instructor)
│   ├── prompter.py         <-- 🟡 PROMPTER: System sklejania szablonów Jinja2
│   ├── tutor_agent.py      <-- ⚪ AGENT: (Główny) Agent AI
│   └── ...
└── ...
```

In [ ]:
%%writefile ai_tutor/engine/prompter.py
from jinja2 import Environment, FileSystemLoader
from config import Config

class SystemPrompter:
    def __init__(self):
        # Jinja2 configuration: we specify the folders where the templates are located
        # This allows us to use {% include 'memory/templates/semantic.j2' %}
        self.env = Environment(loader=FileSystemLoader([
            Config.TEMPLATES_DIR, 
            Config.MEMORY_DIR + "/templates"
        ]))
        
    def build_system_prompt(self, core, profile, session, history):
        template = self.env.get_template("main.j2")
        

        return template.render(
            core=core,
            profile=profile,
            session=session,
            history=history
        )    

## Agent (`engine/tutor_agent.py`)

To jest "Mózg operacji". Jego rola polega na koordynacji: pobiera gotowy prompt z Promptera, wysyła go do Klienta API i odbiera ustrukturyzowaną odpowiedź (`TutorResponse`).

To właśnie tutaj rodzi się **blok 5** (*final messages*). Zobacz na ten fragment kodu – to moment, w którym abstrakcyjne szablony zamieniają się w konkretny payload dla LLM:

```python
messages_payload = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_input}
]
```

### Druga kluczowa rzecz - *Observability*
Zwróć uwagę na metodę `get_debug_state`. To ona "karmi" prawy panel w naszym Tech UI (X-Ray). Nie ma tu żadnej magii – to Ty decydujesz, co chcesz tam widzieć. W tej chwili wrzucamy tam profil, ostatnią myśl i licznik wiadomości. Ale jeśli będziesz chciał(a) śledzić zużycie tokenów, temperaturę modelu czy flagi błędów – po prostu dodajesz to do tego słownika.



### ☝️ Zwróć uwagę
Zwróć uwagę na fragment w `__init__`. Tworzymy tam instancję `StudentProfile` na sztywno ("Marek", poziom B1).

Dlaczego? To klasyczne podejście **MVP**. Zamiast tracić teraz X godzin czy nawet dni na stawianie bazy danych, systemu logowania i endpointów API, po prostu "symulujemy", że użytkownik jest już zalogowany i pobrany. Dzięki temu możemy natychmiast testować logikę agenta. W przyszłości po prostu podmienisz ten fragment na `self.profile = database.get_user(user_id)`. Kod reszty agenta nawet tego nie zauważy.



```text
ai_tutor/
├── app.py                  <-- ⚪ UI (Gradio): Punkt wejścia do aplikacji
├── config.py               <-- 🟢 KONFIGURACJA: Klucze, ścieżki, model
├── engine/
│   ├── client.py           <-- 🟢 KLIENT do API (OpenAI + Instructor)
│   ├── prompter.py         <-- 🟢 PROMPTER: System sklejania szablonów Jinja2
│   ├── tutor_agent.py      <-- 🟡 AGENT: (Główny) Agent AI
│   └── ...
└── ...
```

In [ ]:
%%writefile ai_tutor/engine/tutor_agent.py
from engine.client import get_client
from engine.prompter import SystemPrompter
from config import Config

# schemas
from memory.schemas.core import TutorCoreMemory
from memory.schemas.semantic import StudentProfile, LearningGoal, LearningStyle, CorrectionPolicy
from memory.schemas.working import TutorResponse, SessionState, SessionPhase


class TutorAgent:
    def __init__(self):
        self.prompter = SystemPrompter()

        self.core = TutorCoreMemory()
        
        # --- HARDCODED STATE (MVP) ---
        # At this stage, the profile is defined manually,
        self.profile = StudentProfile(
            name="Marek",
            native_lang="Polish",
            target_lang="English",
            level="B1",
            style=LearningStyle.IMMERSIVE,
            policy=CorrectionPolicy.GENTLE,
            goal=LearningGoal(
                statement="Master Business English for IT",
                target_level="B2"
            )
        )
        
        self.session = SessionState(
            topic="Job Interview Simulations",
            phase=SessionPhase.PRACTICE_MODE
        )
        
        self.messages = [] 
        self.last_thought = ""
        self.client = get_client()

    def run(self, user_input: str) -> str:

        self.messages.append({"role": "user", "content": user_input})
        
        system_prompt = self.prompter.build_system_prompt(
            core=self.core,
            profile=self.profile,
            session=self.session,
            history=self.messages[:-10]
        )
        
        
        messages_payload = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]

    
        response: TutorResponse = self.client.chat.completions.create(
            model=Config.MODEL_NAME,
            messages=messages_payload,
            response_model=TutorResponse
        )
        
        self.last_thought = response.thinking
        self.messages.append({"role": "assistant", "content": response.ai_answer})
        
        return response.ai_answer, system_prompt, self.get_debug_state()
    
    def get_debug_state(self):
        """Metoda dla Tech UI - pozwala podglądać stan agenta"""
        return {
            "core": self.core.model_dump(),
            "profile": self.profile.model_dump(),
            "session": self.session.model_dump(),
            "last_thought": self.last_thought,
            "messages_count": len(self.messages)
        }

## Tech UI (`app.py`)
Na koniec aktualizujemy nasz interfejs Gradio (nasz Tech UI), aby korzystał z nowej klasy `TutorAgent` i wyświetlał proces myślowy (`thinking`).


```text
ai_tutor/
├── app.py                  <-- 🟡 UI (Gradio): Punkt wejścia do aplikacji
├── config.py               <-- 🟢 KONFIGURACJA: Klucze, ścieżki, model
├── engine/
│   ├── client.py           <-- 🟢 KLIENT do API (OpenAI + Instructor)
│   ├── prompter.py         <-- 🟢 PROMPTER: System sklejania szablonów Jinja2
│   ├── tutor_agent.py      <-- 🟢 AGENT: (Główny) Agent AI
│   └── ...
└── ...
```

In [ ]:
%%writefile ai_tutor/app.py
import gradio as gr
from engine.tutor_agent import TutorAgent

agent = TutorAgent()

def interact(message, history):
    if not message: return "", None, None
    
    answer, sys_prompt, debug_state = agent.run(message)
    

    formatted_answer = (
        f"{answer}\n\n"
        f"<details>\n"
        f"\n\n<summary>👇 SYSTEM PROMPT (click) 👇</summary>\n\n"
        f"```text\n{sys_prompt}\n```\n"
        f"</details>"
    )
    
    return formatted_answer, debug_state

# --- LAYOUT ---
with gr.Blocks(title="AI Tutor - Tech UI v0.2", theme=gr.themes.Default()) as demo:
    gr.Markdown("# AI Tutor - Tech UI (v0.2)")
    
    with gr.Row():
        # Left => Chat
        with gr.Column(scale=6):
            
            
            with gr.Accordion("🧠 Live Agent State (Auto-Update)", open=False):
                
                state_json = gr.JSON(label="Internal State")

            # ChatInterface
            chatbot = gr.ChatInterface(
                fn=interact,
                type="messages",
                additional_outputs=[state_json]
            )
            
        # Right => Description
        with gr.Column(scale=4, variant="panel"):
            gr.Markdown("""
            ### 🛠️ Instrukcja
            
            1. **System Prompt:** Pod każdą odpowiedzią bota znajdziesz mały trójkąt ▶ i napis `🛠️ SYSTEM PROMPT`. Kliknij go, aby rozwinąć surowy kontekst, który dostał model.
            2. **Live State:** Poniżej okna czatu znajduje się zwinięty panel `🧠 Live Agent State`. Rozwiń go, jeśli chcesz podejrzeć zmienne w Pythonie (np. `thinking`, `profile`).
            
            ---
            **Test:** Napisz "Cześć"...
            """)

if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=8080)

Dobra, teraz czas na wypełnienie naszej pamięci. Mamy puste pliki, które czekają na logikę.

Pamiętasz lekcję 2.3? Zaprojektowaliśmy tam trójwarstwową strukturę pamięci: **Core (DNA)**, **Semantic (Profil)** i **Working (Sesja)**. Wtedy wszystko "wisiało" w jednym notebooku. Teraz zrobimy to profesjonalnie, rozdzielając definicje (Schemas) od ich reprezentacji (Templates).

Pomogę Ci zacząć. Pokażę Ci, jak wypełnić warstwę **CORE**, żebyś złapać wzorzec. Potem Ty przejmiesz stery i zaimplementujesz resztę.

### Warstwa pamięci *core* (DNA)

Tożsamość agenta składa się z dwóch elementów:
1.  **Schema (`core.py`):** Definicja Pydantic – jakie pola istnieją (name, role, rules).
2.  **Template (`core.j2`):** Szablon Jinja2 – jak te pola mają wyglądać dla modelu LLM.


```text
ai_tutor/
├── memory/
│   ├── schemas/
│   │   ├── core.py         <-- 🟡 CORE: Tożsamość (DNA)
│   │   ├── semantic.py     <-- ⚪ SEMANTIC: Profil Studenta
│   │   └── working.py      <-- ⚪ WORKING: Stan Sesji + Output
│   ├── templates/
│   │   ├── core.j2         <-- 🟡 Wyświetlanie DNA
│   │   ├── semantic.j2     <-- ⚪ Wyświetlanie Profilu
│   │   └── working.j2      <-- ⚪ Wyświetlanie Stanu Sesji
├── templates/
│   └── main.j2             <-- ⚪ MASTER PROMPT (Orchestrator)
│
└── ...
```

Zapiszmy to:

In [ ]:
%%writefile ai_tutor/memory/schemas/core.py
from pydantic import BaseModel
from typing import List

class TutorCoreMemory(BaseModel):
    name: str = "Aoki"
    creator_name: str = "DataWorkshop Lab"
    role: str = "Language Communication Coach"
    prime_directives: List[str] = [
        "NO_HOMEWORK_SOLVING: Never provide full answers immediately.",
        "COMMUNICATION_FIRST: Prioritize fluency over grammar."
    ]

In [ ]:
%%writefile ai_tutor/memory/templates/core.j2
<core_memory>
    <identity>
        Name: {{ core.name }}
        Role: {{ core.role }}
        Created By: {{ core.creator_name }}
    </identity>
    
    <rules>
        {% for rule in core.prime_directives %}
        - {{ rule }}
        {% endfor %}
    </rules>
</core_memory>

Widzisz zależność? Plik `.py` definiuje strukturę danych, a `.j2` "ubiera" ją w tagi XML, które są czytelne dla modelu. Teraz Twoja kolej na resztę warstw.



## 🧠 Zadanie 2.4.1: Definicja Danych (semantic & working Schemas)

Twoim zadaniem jest uzupełnienie plików `memory/schemas/semantic.py` oraz `memory/schemas/working.py`.

O ile `Semantic` to powtórka z lekcji 2.3 (Profile), o tyle w `Working` wprowadzamy kluczową innowację Software 3.0: **Structured Output**.

### **Wymagania:**

1. Semantic (`semantic.py`):
    *   Zdefiniuj `StudentProfile` oraz `LearningGoal`.
    *   Dodaj Enums dla `LearningStyle` i `CorrectionPolicy`.

2. Working (`working.py`) – 
    *   Zdefiniuj `SessionState` (stan rozmowy, np. faza, temat).
    *   **Kluczowe:** Zdefiniuj klasę `TutorResponse`. Nasz agent nie będzie już zwracał "gołego stringa". Będzie zwracał obiekt.
    *   Klasa ta **musi** zawierać pole o nazwie **`ai_answer`** (to w nim znajdzie się tekst dla użytkownika) oraz pole **`thinking`** (dla wewnętrznego monologu - na razie prostsza wersja).
    *   Dlaczego `ai_answer`? Bo takiej nazwy spodziewa się kod `tutor_agent.py`, który wdrożymy później. Jeśli nazwiesz to pole inaczej (np. `content`), agent się wywróci.


```text
ai_tutor/
├── memory/
│   ├── schemas/
│   │   ├── core.py         <-- 🟢 CORE: Tożsamość (DNA)
│   │   ├── semantic.py     <-- 🟡 SEMANTIC: Profil Studenta
│   │   └── working.py      <-- 🟡 WORKING: Stan Sesji + Output
│   ├── templates/
│   │   ├── core.j2         <-- 🟢 Wyświetlanie DNA
│   │   ├── semantic.j2     <-- ⚪ Wyświetlanie Profilu
│   │   └── working.j2      <-- ⚪ Wyświetlanie Stanu Sesji
├── templates/
│   └── main.j2             <-- ⚪ MASTER PROMPT (Orchestrator)
│
└── ...
```    
    
<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈
    </summary>
<p>
    
- Pamiętaj o importach: `Enum`, `List`, `Optional`, `date` i oczywiście `BaseModel` z `pydantic`.
- W `TutorResponse` użyj `Field(..., description="...")`, aby instruować model, co ma wpisać w dane pole. To tutaj dzieje się "Prompting in Pydantic".

</p>
<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć rozwiązanie 👈
    </summary>
<p>

```python
%%writefile ai_tutor/memory/schemas/semantic.py
from enum import Enum
from typing import List, Optional
from pydantic import BaseModel, Field
from datetime import date

class LearningStyle(str, Enum):
    VISUAL = "VISUAL"
    ANALYTICAL = "ANALYTICAL"
    IMMERSIVE = "IMMERSIVE"

class CorrectionPolicy(str, Enum):
    GENTLE = "GENTLE"
    STRICT = "STRICT"

class LearningGoal(BaseModel):
    statement: str
    target_level: str
    deadline: Optional[date] = None
    success_criteria: List[str] = Field(default_factory=list)

class StudentProfile(BaseModel):
    name: str
    native_lang: str
    target_lang: str
    level: str
    goal: LearningGoal
    style: LearningStyle
    policy: CorrectionPolicy
```
    
------------------    
    

```python
%%writefile ai_tutor/memory/schemas/working.py
from enum import Enum
from pydantic import BaseModel, Field

# --- STAN SESJI ---
class SessionPhase(str, Enum):
    HOMEWORK_CHECK = "HOMEWORK_CHECK"
    TOPIC_INTRODUCTION = "TOPIC_INTRODUCTION"
    PRACTICE_MODE = "PRACTICE_MODE"
    FEEDBACK_WRAPUP = "FEEDBACK_WRAPUP"

class SessionState(BaseModel):
    """
    Working Memory - co robimy w tej chwili?
    """
    topic: str
    phase: SessionPhase = SessionPhase.PRACTICE_MODE

# --- STRUKTURA ODPOWIEDZI LLM (Structured Output) ---
class TutorResponse(BaseModel):
    thinking: str = Field(..., description="Wewnętrzny monolog agenta.")
    ai_answer: str = Field(..., description="Finalna wypowiedź do użytkownika.")
```

</p>
</details>
</details>


In [ ]:
%%writefile ai_tutor/memory/schemas/semantic.py

# YOUR CODE HERE

In [ ]:
%%writefile ai_tutor/memory/schemas/working.py

# YOUR CODE HERE

## 🧠 Zadanie 2.4.2: Warstwa prezentacji pamięci (Templates)

Mamy już zdefiniowane struktury danych (Pydantic). Ale LLM nie czyta obiektów Pythona. LLM czyta tekst.

Teraz musimy stworzyć **szablony**, które przetłumaczą nasze surowe dane na instrukcje zrozumiale dla modelu. To tutaj decydujemy, jak Agent "widzi" tę pamięć.

Tworzymy dwa kluczowe pliki w folderze `memory/templates/`:

```text
ai_tutor/
├── memory/
│   ├── schemas/
│   │   ├── core.py         <-- 🟢 CORE: Tożsamość (DNA)
│   │   ├── semantic.py     <-- 🟢 SEMANTIC: Profil Studenta
│   │   └── working.py      <-- 🟢 WORKING: Stan Sesji + Output
│   ├── templates/
│   │   ├── core.j2         <-- 🟢 Wyświetlanie DNA
│   │   ├── semantic.j2     <-- 🟡 Wyświetlanie Profilu
│   │   └── working.j2      <-- 🟡 Wyświetlanie Stanu Sesji
├── templates/
│   └── main.j2             <-- ⚪ MASTER PROMPT (Orchestrator)
│
└── ...
```    

1.  **`semantic.j2` (pamięć semantyczna)** – To jest kartoteka studenta. Tutaj model dowiaduje się, z kim rozmawia. Co ważne, instrukcje korekty (Correction Policy) nie mogą być statyczne. Jeśli user ma ustawione `STRICT`, instrukcja musi brzmieć inaczej niż przy `GENTLE`.
2.  **`working.j2` (pamięć operacyjna)** – To jest sterownik sesji. Tutaj dzieje się "magia" sterowania zachowaniem. W zależności od tego, w jakiej fazie jest sesja (np. `HOMEWORK_CHECK` vs `PRACTICE_MODE`), musisz wstrzyknąć zupełnie inne polecenie (`<active_instruction>`).

#### Wymagania:

1.  **Plik `semantic.j2`:**
    *   Wyświetl podstawowe dane (imię, cel).
    *   Użyj `{% if %}`, aby dynamicznie zmienić instrukcję korekty w zależności od pola `profile.policy`.

2.  **Plik `working.j2`:**
    *   Wyświetl obecny temat i fazę.
    *   Zdefiniuj sekcję `<active_instruction>`, która zmienia się w zależności od `session.phase`. To właśnie ten fragment mówi Agentowi, co ma robić w danej sekundzie (np. "Sprawdzaj zadanie" albo "Prowadź roleplay").

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈
    </summary>
<p>

*   Używaj tagów XML, np. `<student_context>`, `<session_state>`. To pomaga modelowi rozróżnić, co jest faktem o studencie, a co bieżącym zadaniem.
*   Dostęp do pól Enuma w Jinja2 uzyskujesz przez `.value`, np. `{{ profile.style.value }}`.
*   Pamiętaj: Logika biznesowa (czyli "co robić w danej fazie") leży teraz w szablonie `working.j2`, a nie w kodzie Pythona!

</p>
<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć rozwiązanie 👈
    </summary>
<p>

```jinja
%%writefile ai_tutor/memory/templates/semantic.j2
<student_context>
    <basic_info>
        Student: {{ profile.name }}
        Level: {{ profile.level }}
        Lang: {{ profile.native_lang }} -> {{ profile.target_lang }}
    </basic_info>

    <mission_objective>
        Goal: "{{ profile.goal.statement }}"
        Target Level: {{ profile.goal.target_level }}
    </mission_objective>

    <psychometrics>
        Style: {{ profile.style.value }}
        Policy: {{ profile.policy.value }}
        
        {% if profile.policy.value == 'STRICT' %}
        ⚠️ INSTRUCTION: User demands perfection. Correct EVERY mistake.
        {% else %}
        ℹ️ INSTRUCTION: User wants flow. Correct ONLY critical errors.
        {% endif %}
    </psychometrics>
</student_context>
```
----
    
    
```jinja
%%writefile ai_tutor/memory/templates/working.j2
<session_state>
    <status>
        Current Phase: {{ session.phase.value }}
        Current Topic: {{ session.topic }}
    </status>
    
    <active_instruction>
        {% if session.phase.value == 'HOMEWORK_CHECK' %}
        TASK: Verify homework. Do NOT start new topic yet.
        {% elif session.phase.value == 'PRACTICE_MODE' %}
        TASK: Conduct roleplay regarding "{{ session.topic }}". Engage the student.
        {% endif %}
    </active_instruction>
</session_state>
```

</p>
</details>
</details>

In [ ]:
%%writefile ai_tutor/memory/templates/semantic.j2

#YOUR CODE HERE



In [ ]:
%%writefile ai_tutor/memory/templates/working.j2


#YOUR CODE HERE

## 🧠 Zadanie 2.4.3: Orkiestracja (Main Prompt)

Ostatni element układanki. Stwórz plik `templates/main.j2`. To jest główny system prompt, który skleja wszystko w całość.

```text
ai_tutor/
├── memory/
│   ├── schemas/
│   │   ├── core.py         <-- 🟢 CORE: Tożsamość (DNA)
│   │   ├── semantic.py     <-- 🟢 SEMANTIC: Profil Studenta
│   │   └── working.py      <-- 🟢 WORKING: Stan Sesji + Output
│   ├── templates/
│   │   ├── core.j2         <-- 🟢 Wyświetlanie DNA
│   │   ├── semantic.j2     <-- 🟢 Wyświetlanie Profilu
│   │   └── working.j2      <-- 🟢 Wyświetlanie Stanu Sesji
├── templates/
│   └── main.j2             <-- 🟡 MASTER PROMPT (Orchestrator)
│
└── ...
```

#### Wymagania:
1.  Użyj `{% include %}` aby zaimportować szablony: `core.j2`, `semantic.j2` i `working.j2`.
2.  Dodaj pętlę wyświetlającą historię rozmowy.
3.  Zakończ prompt silnym wezwaniem do działania ("Based on <active_instruction>...").

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈
    </summary>
<p>
    
*   Jinja szuka szablonów w ścieżkach zdefiniowanych w `Config`. Możesz odwoływać się do plików po nazwie, jeśli są w ścieżce loadera (np. `include 'core.j2'`), lub relatywnie.
*   Historia rozmowy to lista słowników (`msg.role`, `msg.content`).

</p>
<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć rozwiązanie 👈
    </summary>
<p>

```jinja
%%writefile ai_tutor/templates/main.j2
You are an advanced AI Tutor. 
Adopt the following persona and context to guide the student.

{# 1. CORE DNA #}
{% include 'core.j2' %}

{# 2. STUDENT PROFILE #}
{% include 'semantic.j2' %}

{# 3. SESSION CONTEXT #}
{% include 'working.j2' %}

### CONVERSATION HISTORY ###
{% for msg in history %}
{{ msg.role.upper() }}: {{ msg.content }}
{% endfor %}

Based on <active_instruction> and <psychometrics>, generate the next response.
```

</p>
</details>
</details>

In [ ]:
%%writefile ai_tutor/templates/main.j2

## YOUR CODE HERE

Świetnie. Zakładam, że Twoje pliki w folderach `memory` i `templates` są już uzupełnione.

Jeśli w którymś momencie czujesz, że nie wiesz jak ogarnąć – to normalne. W inżynierii systemów agentowych doświadczenie zdobywa się krok po kroku (np. jedna literówka w nazwie pola w Jinja2 potrafi wyłożyć cały prompt). W takim przypadku śmiało zajrzyj do sekcji z rozwiązaniami powyżej (podpowiedzi). Traktuj je jako dokumentację referencyjną – sprawdź, czy Twoje struktury zgadzają się z wzorcem.

### Czas na integration test

Mamy dane (Schemas), mamy widoki (Templates) i mamy silnik (Engine). Czas to wszystko spiąć i zobaczyć, czy nasz "Mózg" faktycznie działa.

⚠️ **Ważna uwaga operacyjna:**
Uruchomienie poniższej komórki postawi serwer Gradio. To **zablokuje** ten Notebook (zobaczysz gwiazdkę `[*]`). To oczekiwane zachowanie – serwer nasłuchuje.
Jeśli w trakcie testów zauważysz błąd i będziesz chciał(a) poprawić kod w którymś pliku, musisz najpierw **zatrzymać** tę komórkę (przycisk Stop ⏹️ na górze), wprowadzić zmiany i uruchomić ją ponownie.

No to ruszamy!

In [ ]:
!python ai_tutor/app.py

## 👉👉 [Tech UI (Port 8080)](/hub/user-redirect/proxy/8080/) 👈👈

## Podsumowanie

Przeszliśmy drogę od "Notatnika" do "Systemu". Stworzyliśmy strukturę katalogów, która oddziela:
1.  **Tożsamość i pamięć** (`memory/`)
2.  **Mechanikę działania** (`engine/`)
3.  **Umiejętności proceduralne** (`skills/`)
4.  **Interfejs** (`app.py`)


Ta struktura może wydawać się przytłaczająca na początku ("Tyle plików dla prostego bota?"), ale uwierz mi – gdy Twój system urośnie, będziesz sobie dziękować za ten porządek.

### Najważniejsze wnioski:
  1. Kod sam w sobie przestał być nadrzędną wartością, ustępując miejsca przemyślanej architekturze i projektowaniu przepływu danych.
  2. Wdrożono wyraźny podział na Software 1.0 obsługujące klasyczne IT oraz Software 3.0 odpowiedzialne za warstwę agentową i wnioskowanie.
  3. Stworzono fizyczną strukturę plików odzwierciedlającą logiczny podział na pamięć, silnik oraz umiejętności agenta.
  4. Zastosowano warstwę Promptera, który działa jak system renderujący, sklejając szablony i dane w gotowy kontekst dla modelu.
  5. Zdefiniowano strukturę odpowiedzi zawierającą pole thinking, co jest kluczowe dla obserwowalności procesu myślowego modelu (CoT).
  6. Wykorzystano skrypty automatyzujące tworzenie szkieletu projektu, eliminując ryzyko błędów przy ręcznej konfiguracji środowiska.
  7. Przygotowano funkcjonalny system (MVP) ze sprawnym mózgiem (*engine*) i częściowa z pamięcią (*memory*).


## Co dalej?
- W kolejnej lekcji wypełnimy ten szkielet życiem, zanurzając się  w temat pamięci proceduralnej.
- Zaimplementujemy konkretne umiejętności (Skills) oraz uruchomimy pełny cykl życia pamięci, co drastycznie wzmocni jakość i sprawczość naszego agenta.

## Przydatne linki
1. [Gradio](https://www.gradio.app)
2. [Gradio Layout Guide](https://www.gradio.app/guides/controlling-layout)
3. [Gradio Themes](https://www.gradio.app/guides/themes)
4. [How to Create a Custom Chatbot with Gradio Blocks](https://www.gradio.app/guides/creating-a-custom-chatbot-with-blocks)
5. [Gradio awesome-demos](https://github.com/gradio-app/awesome-demos)
6. [Awesome AI Apps](https://github.com/Arindam200/awesome-ai-apps)
7. [Why (Senior) Engineers Struggle to Build AI Agents](https://www.philschmid.de/why-engineers-struggle-building-agents) 